# Metacatalog source quality flags

Attach a **32-bit `quality_flag`** to every merged source. Rows are **not** dropped:
each bit encodes a property or quality concern so downstream cuts can be composed
with `quality_flag == 0` (all checks passed) or `quality_flag & MASK == 0`.

**Convention:** a bit is **0 if that check is good/reliable** and **1** if the
property is a concern. Bits 16–31 are reserved (stay 0).

| Bit | Name | Set (1) when |
|---:|---|---|
| 0 | `HAS_NAN` | a core measurement column is NaN (not per-band extras) |
| 1 | `INVALID_ASTROMETRY` | RA, DEC, or Peak_flux is non-finite (was E0) |
| 2 | `SINGLE_LST` | `n_lst_contributions == 1` |
| 3 | `SINGLE_UNIQUE_BAND` | uniquely associated in exactly one band (`n_assoc==1`; Full/seed band counts) |
| 4 | `UNPHYSICAL_FLUX` | `(Total−Peak)/hypot(E) < −3` (was E2) |
| 5 | `RESID_ABS_FAIL` | island RMS or \|mean\| above absolute Jy/beam cuts (was E3) |
| 6 | `RESID_PCTL_RMS` | `Resid_Isl_rms` outside the catalog 1–99 percentile |
| 7 | `RESID_PCTL_MEAN` | `Resid_Isl_mean` outside the catalog 1–99 percentile |
| 8 | `JITTER_FAIL` | cluster RMS `> 0.3 × BMAJ` (was E4) |
| 9 | `CONFUSED_ASSOC` | any `n_assoc_* > 1` (was E5) |
| 10 | `NO_VLSSR` | no positional match to the VLSSR catalog |
| 11 | `SCODE_COMPLEX` | PyBDSF `S_Code` is `C` or `M` |
| 12 | `LOW_ELEVATION` | elevation at `representative_lst` &lt; 10° |
| 13 | `HIGH_ELLIPTICITY` | `Maj / Min > 3` (source FWHM axes) |
| 14 | `EXTENDED` | `Maj > 3 × BMAJ` |
| 15 | `LARGE_SINGLE` | (`EXTENDED` or `HIGH_ELLIPTICITY`) and (`SINGLE_UNIQUE_BAND` or `SINGLE_LST`) |

Residuals, unphysical flux, and jitter still use the **origin-band seeded LST row**.
α corner cuts are **not** encoded. The parent `metacatalog.parquet` is never overwritten.

Works with **RGB/Full** catalogs (`Full`, `Blue`, `Green`, `Red`) and **frequency-labeled**
subbands (e.g. `18MHz` … `82MHz`; no top-level `Peak_flux` — flux is per-subband; `astrometry_band` is the highest-frequency measurement on each row).

Requires `lwa-catalog[analyze]` (`healpy` for binning, `lwa-healpix` for HiPS export).
The **HiPS sky viewer** at the end also needs `lwa-catalog[viz]` (`panel` + `ipyaladin`).

Set ``REUSE_CACHED_RELIABILITY = True`` to load existing
``metacatalog_quality.parquet`` / ``metacatalog_quality_flags.parquet`` (and existing HiPS
directories) instead of recomputing or writing outputs.

**Run cells in order.** Reload this notebook from disk and restart the kernel if it still
imports `filter_metacatalog_reliability` as the main product.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lwa_catalog.analyze import (
    QualityFlagResult,
    ReliabilityConfig,
    SourceQualityFlag,
    assign_source_quality_flags,
    decode_quality_flag,
    filter_by_quality_mask,
    is_subband_metacatalog,
    load_vlssr_catalog,
    metacatalog_to_healpix,
    quality_flag_legend,
    quality_flag_mask_from_names,
    representative_peak_flux,
    write_healpix_hips,
)
from lwa_catalog.io import (
    discover_lst_merged_bands,
    read_all_lst_merged,
    read_metacatalog,
    read_table,
    seed_band_from_discovery,
    write_table,
)
from lwa_catalog.paths import CatalogLayout

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coaddR-0.75_subband")  # existing fusion tree
NSIDE = 512
REUSE_CACHED_RELIABILITY = False  # load cached Parquet/HiPS; skip all writes when True
# HiPS output directory name under CATALOG_DIR; fields: {name} (full|core_clean|…), {nside}
HIPS_DIR_TEMPLATE = "metacatalog_coaddR-0.75_{name}.hips"
WRITE_SUBSET = False
CONFIG = ReliabilityConfig(
    resid_rms_thresh_jy=1.0,
    resid_mean_thresh_jy=1.0,
    resid_percentile_lo=1.0,
    resid_percentile_hi=99.0,
    jitter_bmaj_frac=0.3,
    min_elevation_deg=10.0,
    max_source_ellipticity=3.0,
)

layout = CatalogLayout(CATALOG_DIR)
BANDS = discover_lst_merged_bands(layout)
SEED_BAND = seed_band_from_discovery(BANDS)
# Subband HiPS maps weight Gaussians by Peak_flux at this band (default: highest-frequency seed).
HIPS_FLUX_BAND = SEED_BAND
# Default exclusion set: drop rows with any of these quality bits set.
EXCLUDE_FLAGS = (
    "HAS_NAN",
    "INVALID_ASTROMETRY",
    "UNPHYSICAL_FLUX",
    "RESID_ABS_FAIL",
    "RESID_PCTL_RMS",
    "RESID_PCTL_MEAN",
    "JITTER_FAIL",
    "LARGE_SINGLE",
)
EXCLUDE_MASK = quality_flag_mask_from_names(EXCLUDE_FLAGS)
# HiPS ``core_clean`` uses the same default exclusion mask.
HIPS_CORE_CLEAN_FLAGS = EXCLUDE_FLAGS
HIPS_CORE_CLEAN_MASK = EXCLUDE_MASK

PATHS = {
    "quality": layout.root / "metacatalog_quality.parquet",
    "flags": layout.root / "metacatalog_quality_flags.parquet",
}
print("CATALOG_DIR =", layout.root.resolve())
print("BANDS =", BANDS)
print("SEED_BAND =", SEED_BAND)
print("REUSE_CACHED_RELIABILITY =", REUSE_CACHED_RELIABILITY)
print("HIPS_DIR_TEMPLATE =", HIPS_DIR_TEMPLATE)
print("EXCLUDE_FLAGS =", EXCLUDE_FLAGS)
print("EXCLUDE_MASK =", EXCLUDE_MASK)
display(quality_flag_legend())


CATALOG_DIR = /fast/claw/metacatalog_coaddR-0.75_subband
BANDS = ('18MHz', '23MHz', '27MHz', '32MHz', '36MHz', '41MHz', '46MHz', '50MHz', '55MHz', '59MHz', '64MHz', '69MHz', '73MHz', '78MHz', '82MHz')
SEED_BAND = 82MHz
REUSE_CACHED_RELIABILITY = False
HIPS_DIR_TEMPLATE = metacatalog_coaddR-0.75_{name}.hips
EXCLUDE_FLAGS = ('HAS_NAN', 'INVALID_ASTROMETRY', 'UNPHYSICAL_FLUX', 'RESID_ABS_FAIL', 'RESID_PCTL_RMS', 'RESID_PCTL_MEAN', 'JITTER_FAIL', 'LARGE_SINGLE')
EXCLUDE_MASK = 33267


,bit,value,name,meaning
0,0,1,HAS_NAN,core measurement column is NaN
1,1,2,INVALID_ASTROMETRY,"RA, DEC, or Peak_flux is non-finite"
2,2,4,SINGLE_LST,n_lst_contributions == 1
3,3,8,SINGLE_UNIQUE_BAND,uniquely associated in exactly one band
4,4,16,UNPHYSICAL_FLUX,(Total-Peak)/hypot(E) < -3
5,5,32,RESID_ABS_FAIL,Resid_Isl_rms or |Resid_Isl_mean| above absolu...
6,6,64,RESID_PCTL_RMS,Resid_Isl_rms outside catalog 1–99 percentile
7,7,128,RESID_PCTL_MEAN,Resid_Isl_mean outside catalog 1–99 percentile
8,8,256,JITTER_FAIL,cluster RA/Dec RMS > 0.3 × BMAJ
9,9,512,CONFUSED_ASSOC,any n_assoc_* > 1


## Load metacatalog + LST-merged cache

In [2]:
# Read fusion metacatalog (not metacatalog_quality.parquet) to compute flags.
metacatalog = read_metacatalog(layout, prefer_quality=False, quality_mask=None)
print(f"metacatalog rows: {len(metacatalog)}")

_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["quality"].is_file()
    and PATHS["flags"].is_file()
)
if _cache_ready:
    lst_merged = None
    vlssr = None
    print("Cache hit for quality flags — skipping LST-merged / VLSSR load")
else:
    lst_merged = read_all_lst_merged(layout, bands=BANDS)
    for band, df in lst_merged.items():
        print(f"  LST {band}: {len(df)} rows")
    try:
        vlssr = load_vlssr_catalog()
        print(f"VLSSR rows: {len(vlssr)}")
    except FileNotFoundError as exc:
        vlssr = None
        print(f"VLSSR not loaded ({exc}); NO_VLSSR bit will be skipped")


metacatalog rows: 138732
  LST 18MHz: 9759 rows
  LST 23MHz: 14463 rows
  LST 27MHz: 20497 rows
  LST 32MHz: 27130 rows
  LST 36MHz: 34596 rows
  LST 41MHz: 49228 rows
  LST 46MHz: 57551 rows
  LST 50MHz: 66173 rows
  LST 55MHz: 74589 rows
  LST 59MHz: 81115 rows
  LST 64MHz: 95244 rows
  LST 69MHz: 99443 rows
  LST 73MHz: 104228 rows
  LST 78MHz: 110217 rows
  LST 82MHz: 93552 rows
VLSSR rows: 92965


In [3]:
metacatalog

,meta_id,origin_band,bands_present,RA,DEC,Maj,Min,PA,DC_Maj,DC_Min,...,source_file_59MHz,source_file_55MHz,source_file_50MHz,source_file_46MHz,source_file_41MHz,source_file_36MHz,source_file_32MHz,source_file_27MHz,source_file_23MHz,source_file_18MHz
0,134592,59MHz,"59MHz,23MHz,18MHz",50.238426,41.700230,1.024100,0.151483,174.693564,0.866252,0.102365,...,59MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,None,None,None,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...
1,7,82MHz,"82MHz,78MHz,73MHz,69MHz,64MHz,59MHz,55MHz,50MH...",49.931915,41.516249,0.112163,0.101283,62.806622,0.066462,0.054576,...,59MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,55MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,50MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,46MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,41MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,36MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...
2,93560,78MHz,"78MHz,69MHz,64MHz,23MHz,18MHz",49.702655,41.296361,0.341893,0.050005,126.855850,0.000000,0.000000,...,None,None,None,None,None,None,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...
3,134527,59MHz,"59MHz,18MHz",68.967301,29.460631,0.197460,0.059312,144.658354,0.000000,0.000000,...,59MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,None,None,None,None,None,None,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...
4,1035,82MHz,"82MHz,78MHz,73MHz,69MHz,64MHz,59MHz,50MHz,36MH...",68.926554,29.690979,0.288427,0.109929,49.281993,0.000000,0.000000,...,59MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,50MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,36MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138727,118997,78MHz,"78MHz,69MHz",162.520709,45.545409,0.113523,0.110951,70.219371,0.069350,0.058975,...,None,None,None,None,None,None,None,None,None,None
138728,127038,73MHz,73MHz,164.132639,14.495232,0.215192,0.107824,172.523311,0.188907,0.039030,...,None,None,None,None,None,None,None,None,None,None
138729,119045,78MHz,78MHz,120.625826,40.288346,0.161483,0.091851,109.006501,0.131524,0.014900,...,None,None,None,None,None,None,None,None,None,None
138730,119047,78MHz,78MHz,177.742437,40.446264,0.134148,0.100703,36.690644,0.094950,0.041982,...,None,None,None,None,None,None,None,None,None,None


## Load or compute quality flags

With ``REUSE_CACHED_RELIABILITY=True``, load ``metacatalog_quality.parquet`` /
``metacatalog_quality_flags.parquet``. If cache files are missing, flags still
run in memory but **no Parquet or HiPS files are written**. Set the flag to
``False`` to regenerate and write outputs.

Every input row is retained. Use ``quality.catalog.query("quality_flag == 0")``
for the all-clear subset, or mask individual bits with
``SourceQualityFlag`` / ``decode_quality_flag``.


In [4]:
def _quality_from_parquet(catalog_path: Path, flags_path: Path) -> QualityFlagResult:
    cat = read_table(catalog_path)
    flags = read_table(flags_path) if flags_path.is_file() else pd.DataFrame()
    return QualityFlagResult(
        catalog=cat,
        flags=flags,
        bit_counts=quality_flag_bit_counts(flags if not flags.empty else cat),
        warnings=[f"loaded from {catalog_path.name}"],
    )


_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["quality"].is_file()
    and PATHS["flags"].is_file()
)
if _cache_ready:
    quality = _quality_from_parquet(PATHS["quality"], PATHS["flags"])
    print(f"Loaded cached quality flags: n={len(quality.catalog)}")
else:
    if REUSE_CACHED_RELIABILITY:
        print("Cache missing — regenerating quality flags…")
    quality = assign_source_quality_flags(
        metacatalog,
        layout,
        config=CONFIG,
        lst_merged=lst_merged,
        vlssr=vlssr,
    )

n = len(quality.catalog)
n_clear = int((quality.catalog["quality_flag"] == 0).sum())
print(f"n_sources = {n}")
print(f"n_all_clear (quality_flag==0) = {n_clear}")
if quality.warnings:
    print("warnings:")
    for w in quality.warnings[:12]:
        print(f"  - {w}")
    if len(quality.warnings) > 12:
        print(f"  … {len(quality.warnings) - 12} more")

display(quality.bit_counts)

reliable = quality.catalog.loc[quality.catalog["quality_flag"] == 0]
flagged = quality.catalog.loc[quality.catalog["quality_flag"] != 0]
core_clean = filter_by_quality_mask(quality.catalog, HIPS_CORE_CLEAN_MASK)
print(
    f"reliable={len(reliable)}  flagged={len(flagged)}  "
    f"core_clean={len(core_clean)}"
)
display(quality.catalog.head(5))
if len(flagged):
    sample = flagged.head(5).copy()
    sample["flag_names"] = sample["quality_flag"].map(
        lambda v: ", ".join(decode_quality_flag(int(v)))
    )
    if is_subband_metacatalog(sample):
        sample["_Peak_flux"] = representative_peak_flux(sample)
        show_cols = ["meta_id", "RA", "DEC", "astrometry_band", "_Peak_flux", "quality_flag", "flag_names"]
    else:
        show_cols = ["meta_id", "RA", "DEC", "Peak_flux", "quality_flag", "flag_names"]
    display(sample[[c for c in show_cols if c in sample.columns]])

# Example: keep sources that are multi-LST and VLSSR-matched, ignoring other bits
# mask = int(SourceQualityFlag.SINGLE_LST | SourceQualityFlag.NO_VLSSR)
# subset = quality.catalog.loc[quality.catalog["quality_flag"] & mask == 0]


n_sources = 138732
n_all_clear (quality_flag==0) = 57403
warnings:
  - using merge-time cluster jitter (skipping per-hour rematch)


,bit,name,n_set,fraction,meaning
0,0,HAS_NAN,0,0.000000,core measurement column is NaN
1,1,INVALID_ASTROMETRY,0,0.000000,"RA, DEC, or Peak_flux is non-finite"
2,2,SINGLE_LST,51616,0.372055,n_lst_contributions == 1
3,3,SINGLE_UNIQUE_BAND,13564,0.097771,uniquely associated in exactly one band
4,4,UNPHYSICAL_FLUX,229,0.001651,(Total-Peak)/hypot(E) < -3
5,5,RESID_ABS_FAIL,1418,0.010221,Resid_Isl_rms or |Resid_Isl_mean| above absolu...
6,6,RESID_PCTL_RMS,2776,0.020010,Resid_Isl_rms outside catalog 1–99 percentile
7,7,RESID_PCTL_MEAN,2776,0.020010,Resid_Isl_mean outside catalog 1–99 percentile
8,8,JITTER_FAIL,1073,0.007734,cluster RA/Dec RMS > 0.3 × BMAJ
9,9,CONFUSED_ASSOC,3426,0.024695,any n_assoc_* > 1


reliable=57403  flagged=81329  core_clean=128352


,meta_id,origin_band,bands_present,RA,DEC,Maj,Min,PA,DC_Maj,DC_Min,...,source_file_55MHz,source_file_50MHz,source_file_46MHz,source_file_41MHz,source_file_36MHz,source_file_32MHz,source_file_27MHz,source_file_23MHz,source_file_18MHz,quality_flag
0,134592,59MHz,"59MHz,23MHz,18MHz",50.238426,41.700230,1.024100,0.151483,174.693564,0.866252,0.102365,...,None,None,None,None,None,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,43204
1,7,82MHz,"82MHz,78MHz,73MHz,69MHz,64MHz,59MHz,55MHz,50MH...",49.931915,41.516249,0.112163,0.101283,62.806622,0.066462,0.054576,...,55MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,50MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,46MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,41MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,36MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,96
2,93560,78MHz,"78MHz,69MHz,64MHz,23MHz,18MHz",49.702655,41.296361,0.341893,0.050005,126.855850,0.000000,0.000000,...,None,None,None,None,None,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,43252
3,134527,59MHz,"59MHz,18MHz",68.967301,29.460631,0.197460,0.059312,144.658354,0.000000,0.000000,...,None,None,None,None,None,None,None,None,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,43124
4,1035,82MHz,"82MHz,78MHz,73MHz,69MHz,64MHz,59MHz,50MHz,36MH...",68.926554,29.690979,0.288427,0.109929,49.281993,0.000000,0.000000,...,None,50MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,36MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,None,None,23MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,18MHz_I_deep_Taper_Robust-0.75_pbcorr_dewarped...,516


,meta_id,RA,DEC,astrometry_band,_Peak_flux,quality_flag,flag_names
0,134592,50.238426,41.700230,59MHz,8.355805,43204,"SINGLE_LST, RESID_PCTL_RMS, RESID_PCTL_MEAN, S..."
1,7,49.931915,41.516249,82MHz,103.593142,96,"RESID_ABS_FAIL, RESID_PCTL_RMS"
2,93560,49.702655,41.296361,78MHz,32.509860,43252,"SINGLE_LST, UNPHYSICAL_FLUX, RESID_ABS_FAIL, R..."
3,134527,68.967301,29.460631,59MHz,70.887063,43124,"SINGLE_LST, UNPHYSICAL_FLUX, RESID_ABS_FAIL, R..."
4,1035,68.926554,29.690979,82MHz,8.368058,516,"SINGLE_LST, CONFUSED_ASSOC"


## Write quality catalog + flags (never overwrite `metacatalog.parquet`)


In [5]:
out = layout.root
paths = PATHS
assert paths["quality"].name != "metacatalog.parquet"

if REUSE_CACHED_RELIABILITY:
    print("Skipping write — REUSE_CACHED_RELIABILITY=True")
    for label, p in paths.items():
        status = "ok" if p.is_file() else "missing"
        print(f"  {label}: {p} ({status})")
else:
    write_table(quality.catalog, paths["quality"])
    write_table(quality.flags, paths["flags"])
    for label, p in paths.items():
        print(f"{label}: {p} ({p.stat().st_size} bytes)")


quality: /fast/claw/metacatalog_coaddR-0.75_subband/metacatalog_quality.parquet (54647365 bytes)
flags: /fast/claw/metacatalog_coaddR-0.75_subband/metacatalog_quality_flags.parquet (6070552 bytes)


## Peak_flux Gaussian HEALPix → HiPS

Paints each source as an **elliptical Gaussian** on an equatorial HEALPix map
(`Peak_flux` amplitude; `Maj`/`Min` FWHM in degrees; `PA` from North toward East),
then writes a **HiPS** tile set with `lwa_healpix.healpix_to_hips` (not FITS).
Use `profile="point"` for single-pixel deposits.

On frequency-labeled subband catalogs, every map uses **`Peak_flux_{HIPS_FLUX_BAND}`**
(default `82MHz` via `SEED_BAND`) as the Gaussian amplitude.

Maps:

- **full** — all rows
- **core_clean** — rows with none of `EXCLUDE_FLAGS` set (`quality_flag & 33267 == 0`)
- **reliable** — `quality_flag == 0` (every check passed)
- **flagged** — `quality_flag != 0`
- **single_lst**, **single_unique_band**, **no_vlssr** — sources with that bit set

Install: `pip install 'lwa-catalog[analyze]'` (`healpy` + `lwa-healpix`).

View with Aladin Lite: serve the output directory over HTTP and open `index.html`.
HiPS output directories must be empty (or new); remove old dirs before re-running.

Skipped when ``REUSE_CACHED_RELIABILITY=True`` (uses existing ``.hips`` directories).


In [6]:
# Peak_flux Gaussian HEALPix → HiPS (via lwa-healpix)
# Install: pip install 'lwa-catalog[analyze]'  (healpy + lwa-healpix)
# HiPS dirs must be empty; delete previous outputs before re-running.

def catalog_for_healpix(cat: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of *cat* with ``Peak_flux`` set for HiPS weighting."""
    out = cat.copy()
    if is_subband_metacatalog(out):
        flux_col = f"Peak_flux_{HIPS_FLUX_BAND}"
        if flux_col not in out.columns:
            msg = f"HiPS flux column missing: {flux_col}"
            raise KeyError(msg)
        out["Peak_flux"] = pd.to_numeric(out[flux_col], errors="coerce")
    return out


out = layout.root
hips_dirs = {}
qf = quality.catalog["quality_flag"].to_numpy(dtype=np.uint32)
bit_cats = (
    ("single_lst", SourceQualityFlag.SINGLE_LST),
    ("single_unique_band", SourceQualityFlag.SINGLE_UNIQUE_BAND),
    ("no_vlssr", SourceQualityFlag.NO_VLSSR),
)
tier_cats = (
    ("full", quality.catalog),
    ("core_clean", core_clean),
    ("reliable", reliable),
    ("flagged", flagged),
    *(
        (name, quality.catalog.loc[(qf & np.uint32(bit)) != 0])
        for name, bit in bit_cats
    ),
)
for name, cat in tier_cats:
    print(f"{name}: {len(cat)} sources")

if REUSE_CACHED_RELIABILITY:
    print("Skipping HiPS export — REUSE_CACHED_RELIABILITY=True")
    for name, _cat in tier_cats:
        hips_dir = out / HIPS_DIR_TEMPLATE.format(name=name, nside=NSIDE)
        hips_dirs[name] = hips_dir
        status = "ok" if (hips_dir / "properties").is_file() else "missing"
        print(f"  {name}: {hips_dir} ({status})")
else:
    for name, cat in tier_cats:
        m = metacatalog_to_healpix(
            catalog_for_healpix(cat),
            nside=NSIDE,
            weight_col="Peak_flux",
            profile="gaussian",  # Maj/Min FWHM (deg), PA N→E; use "point" for single-pixel
        )
        hips_dir = out / HIPS_DIR_TEMPLATE.format(name=name, nside=NSIDE)
        hips_dirs[name] = write_healpix_hips(
            m,
            hips_dir,
            nest=False,
            coord_frame="equatorial",
            threads=True,
            properties={
                "obs_title": f"metacatalog {name} (Peak_flux {HIPS_FLUX_BAND} Gaussians)",
            },
        )
        print(
            f"{name}: peak_max={float(m.max()):.4g} sum={float(m.sum()):.4g} "
            f"→ {hips_dirs[name]}"
        )


full: 138732 sources
core_clean: 128352 sources
reliable: 57403 sources
flagged: 81329 sources
single_lst: 51616 sources
single_unique_band: 13564 sources
no_vlssr: 44247 sources


FileExistsError: HiPS output directory is not empty: /fast/claw/metacatalog_coaddR-0.75_subband/metacatalog_coaddR-0.75_full.hips

## HiPS sky viewer

Interactive **Aladin Lite** view with independent **Catalog** and **HiPS** dropdowns (mix and match).
Band-colored ellipse overlays are on by default. Requires `lwa-catalog[viz]` (`panel` + `ipyaladin`).

**Catalog** lists every ``*.parquet`` under `CATALOG_DIR` (top-level first). Ellipse coloring
still uses `origin_band` / `band` / the filename. Filter tables that have `quality_flag` with
**Quality bits** (empty = no bit filter) and **Bit match**: **Any set**, **All set**, or **None set**.
Bit widgets are disabled when the selected Parquet has no `quality_flag`.

HiPS tiles load in your **web browser** (not the Jupyter kernel). Serve the catalog tree or
individual `.hips` directories over HTTP and set `HIPS_SERVER` to a URL the browser can reach.

Defaults: catalog = `metacatalog_quality.parquet` when present; HiPS = a deep coadd mosaic for
`Blue` (RGB catalogs) or the highest-frequency subband tile (e.g. `82MHz`) when using
frequency-labeled subbands. Adjust **Max sources in view** if the overlay is truncated in crowded fields. The
overlay refreshes when you pan or zoom. For QA screenshots, use `aladin.save_view_as_image(path)`
on the underlying `ipyaladin` widget (see query notebook **Save sky PNG** for an example).


In [ ]:
import json
import threading
import urllib.error
import urllib.request

import astropy.units as u
import panel as pn
import param
from astropy.coordinates import SkyCoord

from lwa_catalog.analyze import filter_by_quality_flags, quality_flag_names
from lwa_catalog.io import read_table
from lwa_catalog.viz.aladin import overlay_catalog_by_band

# HiPS tiles load in your web browser (not the Jupyter kernel).
# HIPS_LIST_SERVER — survey names from ``cgi-bin/list-hips.py`` (dropdown; kernel fetch).
# HIPS_SERVER — tile base URL passed to Aladin (browser fetch; can differ from list host).
# If the list fetch fails, scans CATALOG_DIR for HiPS ``properties`` directories.
# Example: serve coadd tiles locally:
#   python -m http.server 8000 --directory /fast/claw/metacatalog_coadd2
#   HIPS_LIST_SERVER = "http://localhost:8000"
#   HIPS_SERVER = "http://localhost:8000"
HIPS_LIST_SERVER = "http://lwacalim10:3005"  # or http://localhost:3005 when SSH-tunneled
HIPS_SERVER = "http://localhost:3005"


def default_hips_survey_candidate(bands: tuple[str, ...]) -> str:
    """Preferred HiPS survey stem for RGB/Full or frequency-labeled catalogs."""
    rgb_suffix = (
        "_I_deep_Taper_Robust-0.75_pbcorr_coadd_mosaic_jysr_feather_allsky_"
        "20241218-20250823_N023.hips"
    )
    if "Blue" in bands:
        return f"Blue{rgb_suffix}"
    mhz = sorted(
        (band for band in bands if band.endswith("MHz")),
        key=lambda band: int(band.removesuffix("MHz")),
    )
    if mhz:
        pick = mhz[-1]  # highest-frequency subband present in catalog
        return f"{pick}{rgb_suffix}"
    return f"Blue{rgb_suffix}"


DEFAULT_HIPS_SURVEY = default_hips_survey_candidate(BANDS)
SKY_FOV_DEG = 10.0
HIPS_VIEW_HEIGHT = 700
OVERLAY_MAX_SOURCES_DEFAULT = 1000
OVERLAY_MAX_SOURCES_BOUNDS = (100, 2000)

QUALITY_BIT_OPTIONS = quality_flag_names()
QUALITY_MATCH_OPTIONS = {"Any set": "any", "All set": "all", "None set": "none"}
QUALITY_MATCH_LABELS = {code: label for label, code in QUALITY_MATCH_OPTIONS.items()}

pn.extension(throttled=True)


def discover_local_hips_surveys(catalog_dir: Path = layout.root) -> list[str]:
    """Find HiPS tile directories under ``catalog_dir`` (dirs containing ``properties``)."""
    root = Path(catalog_dir)
    if not root.is_dir():
        return []
    return sorted(
        child.name
        for child in root.iterdir()
        if child.is_dir() and (child / "properties").is_file()
    )


def fetch_hips_surveys(
    list_base: str = HIPS_LIST_SERVER,
    *,
    catalog_dir: Path = layout.root,
    timeout: float = 5.0,
) -> list[str]:
    """Return HiPS survey names from ``list_base``, with local catalog fallback."""
    local = discover_local_hips_surveys(catalog_dir)
    url = f"{list_base.rstrip('/')}/cgi-bin/list-hips.py"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            remote = [str(s) for s in json.loads(resp.read().decode())]
    except (OSError, urllib.error.URLError, json.JSONDecodeError, TimeoutError) as exc:
        print(f"Warning: could not fetch HiPS surveys from {url}: {exc}")
        if local:
            print(
                f"Using {len(local)} HiPS director{'y' if len(local) == 1 else 'ies'} "
                f"under {catalog_dir}"
            )
            return sorted({DEFAULT_HIPS_SURVEY, *local})
        print(f"Falling back to DEFAULT_HIPS_SURVEY={DEFAULT_HIPS_SURVEY!r}")
        return [DEFAULT_HIPS_SURVEY]
    return sorted({DEFAULT_HIPS_SURVEY, *remote, *local})


def default_hips_survey(surveys: list[str]) -> str:
    """Pick a sensible default HiPS survey from the server list."""
    if DEFAULT_HIPS_SURVEY in surveys:
        return DEFAULT_HIPS_SURVEY
    default_name = Path(DEFAULT_HIPS_SURVEY).name
    hit = next((s for s in surveys if default_name in s or Path(s).name == default_name), None)
    if hit is not None:
        return hit
    return surveys[0] if surveys else DEFAULT_HIPS_SURVEY


def hips_survey_url(survey: str, *, base: str = HIPS_SERVER) -> str:
    """Build the HiPS root URL passed to ipyaladin (trailing slash for Aladin Lite)."""
    survey = survey.strip().strip("/")
    if survey.startswith("http://") or survey.startswith("https://"):
        return survey if survey.endswith("/") else f"{survey}/"
    return f"{base.rstrip('/')}/{survey}/"


def parse_coordinate(text: str) -> SkyCoord:
    """Parse a single sky position from free-form text."""
    text = text.strip()
    if not text:
        raise ValueError("Coordinate string is empty")
    parts = text.replace(",", " ").split()
    if len(parts) == 2:
        try:
            ra = float(parts[0])
            dec = float(parts[1])
            return SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
        except ValueError:
            pass
    return SkyCoord(text, frame="icrs")


def discover_catalog_parquets(catalog_dir: Path = layout.root) -> list[str]:
    """Relative paths of ``*.parquet`` files under ``catalog_dir`` (top-level first)."""
    root = Path(catalog_dir)
    if not root.is_dir():
        return []
    rels = [
        str(path.relative_to(root))
        for path in root.rglob("*.parquet")
        if path.is_file()
    ]
    top = sorted(name for name in rels if "/" not in name)
    nested = sorted(name for name in rels if name not in top)
    return top + nested


def default_catalog_parquet(options: list[str]) -> str:
    """Prefer the in-tree quality catalog, then ``metacatalog.parquet``."""
    preferred = []
    quality_path = PATHS.get("quality")
    if quality_path is not None:
        try:
            preferred.append(str(Path(quality_path).relative_to(layout.root)))
        except ValueError:
            preferred.append(Path(quality_path).name)
    preferred.extend(["metacatalog_quality.parquet", "metacatalog.parquet"])
    for name in preferred:
        if name in options:
            return name
    return options[0] if options else "metacatalog.parquet"


CATALOG_KEYS: list[str] = discover_catalog_parquets() or ["metacatalog.parquet"]
DEFAULT_CATALOG = default_catalog_parquet(CATALOG_KEYS)
print(f"overlay catalogs: {len(CATALOG_KEYS)} parquet files under {layout.root}")
print(f"DEFAULT_CATALOG = {DEFAULT_CATALOG}")
print(f"DEFAULT_HIPS_SURVEY = {DEFAULT_HIPS_SURVEY}")


def catalog_overlay_name(catalog_key: str) -> str:
    """Parquet stem passed to overlay band resolution."""
    return Path(catalog_key).stem


def load_catalog(catalog_key: str, cache: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Load a Parquet catalog, reusing the in-memory quality table when it matches."""
    if catalog_key in cache:
        return cache[catalog_key]
    rel = str(catalog_key)
    quality_path = PATHS.get("quality")
    quality_rel = None
    if quality_path is not None:
        try:
            quality_rel = str(Path(quality_path).relative_to(layout.root))
        except ValueError:
            quality_rel = Path(quality_path).name
    if quality_rel is not None and rel in {quality_rel, Path(quality_path).name}:
        df = quality.catalog
    else:
        path = (layout.root / rel).resolve()
        if not path.is_file():
            msg = f"Catalog Parquet not found: {path}"
            raise FileNotFoundError(msg)
        df = read_table(path)
    cache[catalog_key] = df
    return df


def overlay_bit_filter_applies(df: pd.DataFrame) -> bool:
    """True when the selected table carries ``quality_flag``."""
    return "quality_flag" in df.columns


def filter_overlay_catalog(
    df: pd.DataFrame,
    bit_names: list[str],
    match: str,
) -> pd.DataFrame:
    """Apply the quality-bit overlay filter; no-op without flags or empty bits."""
    if not overlay_bit_filter_applies(df) or not bit_names:
        return df
    return filter_by_quality_flags(df, bit_names, match=match)


def overlay_filter_status(df: pd.DataFrame, bit_names: list[str], match: str) -> str:
    """Short overlay-status clause describing the active bit filter."""
    if not overlay_bit_filter_applies(df):
        return "bit filter off (no quality_flag)"
    if not bit_names:
        return "no bit filter"
    bits = ", ".join(bit_names)
    return f"{QUALITY_MATCH_LABELS.get(match, match)} [{bits}]"


class ReliabilityHiPSViewer(pn.viewable.Viewer):
    """Mix-and-match catalog + HiPS sky view with bitmask-filtered overlays."""

    catalog = param.Selector(default=DEFAULT_CATALOG, objects=CATALOG_KEYS)
    show_overlay = param.Boolean(default=True, doc="Draw catalog sources on the HiPS view")
    max_sources = param.Integer(
        default=OVERLAY_MAX_SOURCES_DEFAULT,
        bounds=OVERLAY_MAX_SOURCES_BOUNDS,
        doc="Max sources drawn inside the current FOV",
    )
    coordinate = param.String(
        default="83.633 -5.391",
        doc="Center coordinate (decimal deg or sexagesimal)",
    )

    def __init__(self, **params):
        super().__init__(**params)
        self._catalog_cache: dict[str, pd.DataFrame] = {}
        self._hips_current_survey = ""
        self._overlay_refresh_timer: threading.Timer | None = None
        self._ready = False

        hips_surveys = fetch_hips_surveys()
        hips_default = default_hips_survey(hips_surveys)
        init_coord = parse_coordinate(self.coordinate)

        from ipyaladin import Aladin

        self._catalog_w = pn.widgets.Select.from_param(
            self.param.catalog, name="Catalog", sizing_mode="stretch_width"
        )
        self._hips_survey_w = pn.widgets.Select(
            name="HiPS survey",
            options=hips_surveys,
            value=hips_default,
            sizing_mode="stretch_width",
        )
        self._bits_w = pn.widgets.MultiChoice(
            name="Quality bits",
            options=QUALITY_BIT_OPTIONS,
            value=[],
            solid=False,
            sizing_mode="stretch_width",
        )
        self._match_w = pn.widgets.RadioBoxGroup(
            name="Bit match",
            options=QUALITY_MATCH_OPTIONS,
            value="any",
            inline=True,
        )
        self._hips_fov_w = pn.widgets.FloatSlider(
            name="FOV (deg)",
            start=1,
            end=90.0,
            value=SKY_FOV_DEG,
            step=0.05,
        )
        self._overlay_w = pn.widgets.Checkbox.from_param(
            self.param.show_overlay,
            name="Show catalog overlay",
        )
        self._max_sources_w = pn.widgets.IntInput.from_param(
            self.param.max_sources,
            name="Max sources in view",
            width=160,
        )
        self._coord_w = pn.widgets.TextInput.from_param(
            self.param.coordinate, name="Coordinate", placeholder="RA Dec"
        )
        self._center_btn = pn.widgets.Button(name="Center view", button_type="primary")
        self._center_btn.on_click(self._on_center_click)

        self._hips_status = pn.pane.Markdown("", sizing_mode="stretch_width")
        self._overlay_status = pn.pane.Markdown("", sizing_mode="stretch_width")

        self._hips_current_survey = hips_survey_url(hips_default)
        self._aladin = Aladin(
            survey=self._hips_current_survey,
            target=init_coord,
            fov=SKY_FOV_DEG,
            height=HIPS_VIEW_HEIGHT,
        )
        self._hips_view = pn.pane.IPyWidget(
            self._aladin,
            height=HIPS_VIEW_HEIGHT + 20,
            sizing_mode="stretch_width",
        )
        self._aladin.observe(self._on_aladin_view_trait, names=["_target", "_fov"])

        self._catalog_w.param.watch(self._on_catalog_change, "value")
        self._hips_survey_w.param.watch(self._on_hips_survey_change, "value")
        self._hips_fov_w.param.watch(self._on_hips_fov_change, "value")
        self._overlay_w.param.watch(self._on_overlay_change, "value")
        self._max_sources_w.param.watch(self._on_max_sources_change, "value")
        self._bits_w.param.watch(self._on_bit_filter_change, "value")
        self._match_w.param.watch(self._on_bit_filter_change, "value")

        self._panel = pn.Column(
            pn.pane.Markdown("### Sky context (HiPS + catalog overlay)", disable_anchors=True),
            pn.Row(self._catalog_w, self._hips_survey_w),
            self._bits_w,
            pn.Row(self._match_w, self._overlay_w, self._max_sources_w),
            pn.Row(self._hips_fov_w, self._coord_w, self._center_btn),
            self._hips_status,
            self._overlay_status,
            self._hips_view,
        )

        self._ready = True
        self._sync_bit_widgets()
        self._update_sky_view(init_coord)

    def __panel__(self):
        return self._panel

    def _view_coord(self) -> SkyCoord:
        return parse_coordinate(self.coordinate)

    def _overlay_view(self) -> tuple[SkyCoord, float]:
        """Current Aladin center and FOV for viewport-limited overlays."""
        target = self._aladin.target
        if isinstance(target, SkyCoord):
            coord = target
        else:
            ra, dec = target
            coord = SkyCoord(ra=ra, dec=dec, frame="icrs")
        fov = float(self._aladin.fov.to(u.deg).value)
        return coord, fov

    def _on_aladin_view_trait(self, _change) -> None:
        if not self._ready:
            return
        self._schedule_overlay_refresh()

    def _schedule_overlay_refresh(self) -> None:
        timer = self._overlay_refresh_timer
        if timer is not None:
            timer.cancel()
        self._overlay_refresh_timer = threading.Timer(0.35, self._refresh_overlay_from_aladin)
        self._overlay_refresh_timer.daemon = True
        self._overlay_refresh_timer.start()

    def _refresh_overlay_from_aladin(self) -> None:
        if not self._ready or not self.show_overlay:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _set_hips_survey(self, survey_name: str) -> None:
        url = hips_survey_url(survey_name)
        if url == self._hips_current_survey:
            return
        self._aladin.survey = url
        self._hips_current_survey = url

    def _refresh_overlay(
        self,
        coord: SkyCoord | None = None,
        *,
        fov_deg: float | None = None,
    ) -> None:
        view_coord, view_fov = self._overlay_view()
        if coord is None:
            coord = view_coord
        if fov_deg is None:
            fov_deg = view_fov

        catalog_key = str(self.catalog)
        catalog_name = catalog_overlay_name(catalog_key)
        fov = float(fov_deg)

        if not self.show_overlay:
            overlay_catalog_by_band(
                self._aladin,
                pd.DataFrame(),
                catalog_name,
                coord,
                fov,
                replace=True,
            )
            self._overlay_status.object = "_Catalog overlay off._"
            return

        try:
            df = load_catalog(catalog_key, self._catalog_cache)
        except (FileNotFoundError, KeyError) as exc:
            self._overlay_status.object = f"**Overlay failed:** `{exc}`"
            return

        bit_names = [str(n) for n in (self._bits_w.value or [])]
        match = str(self._match_w.value)
        df = filter_overlay_catalog(df, bit_names, match)

        result = overlay_catalog_by_band(
            self._aladin,
            df,
            catalog_name,
            coord,
            fov,
            max_rows=int(self.max_sources),
        )
        cap_note = f" (capped at {self.max_sources})" if result.truncated else ""
        cat_label = catalog_key
        filt = overlay_filter_status(df, bit_names, match)
        self._overlay_status.object = (
            f"**Overlay:** {result.drawn} drawn, {result.in_fov} in FOV{cap_note} "
            f"— {cat_label}; {filt} on `{self._hips_survey_w.value}`."
        )

    def _update_sky_view(self, coord: SkyCoord) -> None:
        self._set_hips_survey(str(self._hips_survey_w.value))
        self._aladin.target = coord
        self._aladin.fov = float(self._hips_fov_w.value)
        self._hips_status.object = (
            f"Centered on RA={coord.ra.deg:.6f}, Dec={coord.dec.deg:.6f} "
            f"(FOV={self._hips_fov_w.value:.2f}°)."
        )
        self._refresh_overlay(coord)

    def _sync_bit_widgets(self) -> None:
        try:
            df = load_catalog(str(self.catalog), self._catalog_cache)
            enabled = overlay_bit_filter_applies(df)
        except (FileNotFoundError, KeyError, OSError):
            enabled = False
        self._bits_w.disabled = not enabled
        self._match_w.disabled = not enabled

    def _on_catalog_change(self, _event=None) -> None:
        if not self._ready:
            return
        self._sync_bit_widgets()
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_bit_filter_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_hips_survey_change(self, _event=None) -> None:
        self._set_hips_survey(str(self._hips_survey_w.value))
        try:
            self._refresh_overlay()
        except Exception:
            pass

    def _on_hips_fov_change(self, _event=None) -> None:
        self._aladin.fov = float(self._hips_fov_w.value)
        try:
            self._refresh_overlay()
        except Exception:
            pass

    def _on_overlay_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_max_sources_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_center_click(self, _event=None) -> None:
        try:
            coord = parse_coordinate(self.coordinate)
        except Exception as exc:
            self._hips_status.object = f"**Center failed:** `{exc}`"
            return
        self._update_sky_view(coord)


hips_viewer = ReliabilityHiPSViewer()
hips_viewer

## Make subsets

In [ ]:
from lwa_catalog.analyze import quality_flag_mask_from_names
from lwa_catalog.io import write_table

In [ ]:
# Edit this list to compose a bitmask; rows with any listed bit set are dropped.
EXCLUDE_FLAGS = [
    "HAS_NAN",
    "INVALID_ASTROMETRY",
    "UNPHYSICAL_FLUX",
    "RESID_ABS_FAIL",
    "RESID_PCTL_RMS",
    "RESID_PCTL_MEAN",
    "JITTER_FAIL",
    "LARGE_SINGLE",
]
mask = quality_flag_mask_from_names(EXCLUDE_FLAGS)
print("EXCLUDE_FLAGS =", EXCLUDE_FLAGS)
print("mask =", mask)

In [ ]:
mask

In [ ]:
if WRITE_SUBSET:
    kept = metacatalog.loc[(quality.catalog['quality_flag'] & mask) == 0]
    out = layout.root / f"metacatalog_resid_mask{mask}.parquet"
    assert out.name != "metacatalog.parquet"
    write_table(kept, out)
    print(f"wrote {len(kept)} / {len(quality.catalog)} → {out}")